In [ ]:
import pandas as pd
import numpy as np
import gymnasium as gym
from gymnasium import spaces
import torch
import torch.nn as nn
from stable_baselines3.common.torch_layers import BaseFeaturesExtractor
from stable_baselines3.common.vec_env import DummyVecEnv
from stable_baselines3 import PPO

In [30]:
# 1. ĐỌC VÀ CHUẨN BỊ DỮ LIỆU
print("Đang nạp dữ liệu...")
raw_df = pd.read_parquet(r"C:\Users\ADMIN\Desktop\Kaggle\output\hmm_v3_op1_extended\master_drl_ready_full.parquet")

# Xoay dữ liệu lấy Ma trận Lợi nhuận (Kích thước: Time x 46 Tickers)
returns_df = raw_df.pivot(index='time', columns='ticker', values='log_return')

# Xoay dữ liệu lấy TẤT CẢ các xác suất trạng thái của TỪNG TICKER
prob0_df = raw_df.pivot(index='time', columns='ticker', values='prob_ticker_0')
prob1_df = raw_df.pivot(index='time', columns='ticker', values='prob_ticker_1')
prob2_df = raw_df.pivot(index='time', columns='ticker', values='prob_ticker_2')

# GỘP tất cả ma trận xác suất (Time x 184)
ticker_regime_probs = pd.concat([prob0_df, prob1_df, prob2_df], axis=1)

# Cân bằng số ngày
returns_df, ticker_regime_probs = returns_df.align(ticker_regime_probs, axis=0, join='inner')
weights_dim = returns_df.shape[1]

print("Returns matrix shape:", returns_df.shape) 
print("Ticker Regime Probs shape:", ticker_regime_probs.shape)

Đang nạp dữ liệu...
Returns matrix shape: (2382, 46)
Ticker Regime Probs shape: (2382, 138)


In [31]:
# 2. XÂY DỰNG MÔI TRƯỜNG GIAO DỊCH 2D
class RegimeAwarePortfolioEnv2D(gym.Env):
    def __init__(self, returns_df, regime_probs, weights_dim, horizon=30, cost_rate=0.001):
        super().__init__()
        self.returns_df = returns_df.reset_index(drop=True)
        self.regime_probs = regime_probs.reset_index(drop=True)
        self.weights_dim = weights_dim
        self.horizon = horizon
        self.cost_rate = cost_rate

        self.action_space = spaces.Box(low=0, high=1, shape=(weights_dim,), dtype=np.float32)
        
        # Observation Box 2D: (46 tickers, 5 features)
        self.observation_space = spaces.Box(
            low=-np.inf, high=np.inf,
            shape=(weights_dim, 4),
            dtype=np.float32
        )

        self.current_step = 0
        self.prev_action = np.ones(weights_dim) / weights_dim

    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        self.current_step = 0
        self.prev_action = np.ones(self.weights_dim) / self.weights_dim
        return self._get_observation(), {}

    def step(self, action):
        action = np.clip(action, 0, 1)
        if action.sum() == 0:
            action = np.ones_like(action) / len(action)
        else:
            action /= action.sum()

        returns = self.returns_df.iloc[self.current_step].values
        reward = np.dot(action, returns)

        transaction_cost = np.sum(np.abs(action - self.prev_action)) * self.cost_rate
        reward -= transaction_cost
        self.prev_action = action

        reward = np.clip(reward, -0.03, 0.03)
        log_return = np.log(1 + reward)

        self.current_step += 1
        terminated = self.current_step >= self.horizon
        truncated = False
        
        if terminated:
            self.current_step -= 1
            obs = self._get_observation()
            self.current_step += 1
        else:
            obs = self._get_observation()
            
        info = {"weights": action, "reward": reward}
        return obs, log_return, terminated, truncated, info

    def _get_observation(self):
        # returns shape: (46, 1)
        returns = self.returns_df.iloc[self.current_step].values.reshape(-1, 1)
        
        # regime shape: (46, 4)
        regime_flat = self.regime_probs.iloc[self.current_step].values
        regime_2d = regime_flat.reshape(3, self.weights_dim).T
        
        # concat shape: (46, 5)
        obs = np.concatenate([returns, regime_2d], axis=1).astype(np.float32)
        return obs

In [32]:
# 3. THIẾT KẾ BỘ NÃO AI (TICKER-LEVEL EXTRACTOR)
class TickerLevelExtractor(BaseFeaturesExtractor):
    def __init__(self, observation_space: spaces.Box, features_dim: int = 128):
        super().__init__(observation_space, features_dim)
        
        self.n_tickers = observation_space.shape[0] # 46
        self.n_features_per_ticker = observation_space.shape[1] # 5
        
        # Mạng nơ-ron phân tích cục bộ
        self.ticker_net = nn.Sequential(
            nn.Linear(self.n_features_per_ticker, 16),
            nn.ReLU(),
            nn.Linear(16, 16),
            nn.ReLU()
        )
        
        # Mạng nơ-ron tổng hợp toàn danh mục
        self.global_net = nn.Sequential(
            nn.Linear(self.n_tickers * 16, features_dim),
            nn.ReLU()
        )

    def forward(self, observations: torch.Tensor) -> torch.Tensor:
        batch_size = observations.shape[0]
        
        obs_reshaped = observations.view(-1, self.n_features_per_ticker)
        ticker_features = self.ticker_net(obs_reshaped)
        
        ticker_features = ticker_features.view(batch_size, -1)
        return self.global_net(ticker_features)

In [33]:
# 4. HUẤN LUYỆN VÀ LƯU MÔ HÌNH
policy_kwargs = dict(
    features_extractor_class=TickerLevelExtractor,
    features_extractor_kwargs=dict(features_dim=128),
)

# Đóng gói môi trường
env_2d = DummyVecEnv([lambda: RegimeAwarePortfolioEnv2D(returns_df, ticker_regime_probs, weights_dim)])

# Khởi tạo mô hình PPO với bộ não mới
print("Đang khởi tạo mô hình AI cấu trúc phân chia Ticker-level...")
model_grouped = PPO("MlpPolicy", env_2d, policy_kwargs=policy_kwargs, verbose=1, n_steps=2048)

# Bắt đầu học 50,000 bước (Bạn có thể tăng số này lên nếu muốn)
print("Bắt đầu huấn luyện...")
model_grouped.learn(total_timesteps=50000)

# Lưu mô hình
model_grouped.save("ppo_grouped_tickers")
print("Đã lưu mô hình thành công!")

Đang khởi tạo mô hình AI cấu trúc phân chia Ticker-level...
Using cpu device
Bắt đầu huấn luyện...
-----------------------------
| time/              |      |
|    fps             | 1442 |
|    iterations      | 1    |
|    time_elapsed    | 1    |
|    total_timesteps | 2048 |
-----------------------------
----------------------------------------
| time/                   |            |
|    fps                  | 934        |
|    iterations           | 2          |
|    time_elapsed         | 4          |
|    total_timesteps      | 4096       |
| train/                  |            |
|    approx_kl            | 0.03145577 |
|    clip_fraction        | 0.278      |
|    clip_range           | 0.2        |
|    entropy_loss         | -65.2      |
|    explained_variance   | -8.44      |
|    learning_rate        | 0.0003     |
|    loss                 | -0.071     |
|    n_updates            | 10         |
|    policy_gradient_loss | -0.046     |
|    std                  | 0.998  

In [34]:
# ==========================================
# 5. ĐÁNH GIÁ (TEST) MÔ HÌNH SAU KHI TRAIN
# ==========================================
import numpy as np
import pandas as pd

try:
    print("Đang nạp mô hình đã lưu...")
    model_test = PPO.load("ppo_grouped_tickers")

    # Lấy dữ liệu của ngày CŨNG CÙNG trong tập dữ liệu để thử nghiệm
    returns_today = returns_df.iloc[-1].values.reshape(-1, 1)
    
    # Ở đây chúng ta có 3 cột xác suất (prob 0, 1, 2)
    regime_today_flat = ticker_regime_probs.iloc[-1].values
    regime_today_2d = regime_today_flat.reshape(3, weights_dim).T

    # Ghép lại thành ma trận (46, 4) y hệt như hàm _get_observation()
    real_observation = np.concatenate([returns_today, regime_today_2d], axis=1).astype(np.float32)

    print("Yêu cầu AI tư vấn danh mục...")
    raw_action, _states = model_test.predict(real_observation, deterministic=True)

    # Chuẩn hóa để tổng tỷ trọng bằng 1 (tức 100% vốn)
    raw_action = np.clip(raw_action, 0, 1)
    if raw_action.sum() == 0:
        final_action = np.ones_like(raw_action) / len(raw_action)
    else:
        final_action = raw_action / raw_action.sum()

    # In ra báo cáo dễ nhìn
    result_df = pd.DataFrame({
        'Cổ phiếu': returns_df.columns,
        'Tỷ trọng phân bổ (%)': (final_action * 100).round(2)
    })

    # Lọc ra những mã được AI ưu ái chia tiền nhiều nhất (> 1%)
    top_picks = result_df[result_df['Tỷ trọng phân bổ (%)'] > 1.0].sort_values(by='Tỷ trọng phân bổ (%)', ascending=False)
    
    print("\n=== DANH MỤC AI KHUYÊN ĐẦU TƯ (LỚN HƠN 1% VỐN) ===")
    print(top_picks.to_string(index=False))
except Exception as e:
    print("Vui lòng chạy ô số 4 (Train mô hình) trước khi test nhé! Lỗi:", e)


Đang nạp mô hình đã lưu...
Yêu cầu AI tư vấn danh mục...

=== DANH MỤC AI KHUYÊN ĐẦU TƯ (LỚN HƠN 1% VỐN) ===
Cổ phiếu  Tỷ trọng phân bổ (%)
     HSG             16.719999
     DXG              9.930000
     NKG              9.840000
     MWG              8.450000
     HCM              8.390000
     REE              8.040000
     STB              7.750000
     PHR              7.110000
     HAG              4.880000
     VCB              4.160000
     SJS              3.380000
     SSI              2.830000
     BID              2.490000
     CTD              1.840000
     SBT              1.680000
     VHC              1.400000
